<a href="https://colab.research.google.com/github/sahebnag/basic-RAG-pipeline/blob/main/HR_Assistant_Nestle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crafting an AI-Powered HR Assistant
## A Use Case for Nestlé's HR Policy Documents

This notebook implements a **Retrieval-Augmented Generation (RAG)** chatbot that answers questions about Nestlé's HR policy.

**Built on the current (2026) LangChain 1.x / Gradio 6.x stack** — every API call below has been smoke-tested against:
* `langchain 1.3.x`
* `langchain-openai 1.2.x`
* `langchain-chroma 1.1.x`
* `langchain-community 0.4.x`
* `langchain-text-splitters 1.1.x`
* `pypdf 6.x`
* `gradio 6.x`
* `huggingface_hub 1.x`

**Workflow**
1. Install the required libraries (latest versions).
2. Configure the OpenAI API key.
3. Load and split Nestlé's HR policy PDF using `PyPDFLoader`.
4. Create vector embeddings with `OpenAIEmbeddings` and store them in a `Chroma` vector database.
5. Build an LCEL RAG chain on top of GPT-3.5 Turbo with a custom prompt.
6. Wrap everything in a Gradio chatbot UI.

**Run order:** execute the cells top-to-bottom. After cell 1 finishes, restart the runtime if Colab prompts you to.

## 1. Install dependencies

We only install the packages we actually need and let pip resolve them against Colab's current stack. Nothing pinned to old versions.

In [ ]:
%pip install -q -U \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-community \
    langchain-text-splitters \
    pypdf \
    gradio

## 2. Import libraries

Note the new LangChain 1.x import path: `langchain_text_splitters` (no longer `langchain.text_splitter`).

In [ ]:
import os
import getpass
import urllib.request

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import gradio as gr

print(f"Gradio version: {gr.__version__}")
print("All libraries imported successfully.")

## 3. Configure the OpenAI API key

Get your key from <https://platform.openai.com/api-keys>. We read it with `getpass` so the value is not stored in the notebook output.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OpenAI API key configured.")

## 4. Download Nestlé's HR policy PDF

We use the official 2012 *Nestlé Human Resources Policy* document published on nestle.com. A `User-Agent` header is set because the server rejects requests with Python's default UA. If the URL ever stops responding, upload your own PDF to the Colab session (click the folder icon on the left) and update `PDF_PATH` accordingly.

In [ ]:
PDF_URL = "https://www.nestle.com/sites/default/files/asset-library/documents/jobs/the_nestle_hr_policy_pdf_2012.pdf"
PDF_PATH = "nestle_hr_policy.pdf"

if not os.path.exists(PDF_PATH):
    print("Downloading Nestlé HR policy PDF ...")
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(PDF_PATH, "wb") as f:
        f.write(response.read())

print(f"PDF available at: {PDF_PATH} ({os.path.getsize(PDF_PATH) / 1024:.1f} KB)")

## 5. Load the PDF and split it into chunks

`PyPDFLoader` returns one `Document` per page. We then break the pages into smaller, overlapping chunks so that semantically related sentences stay together when retrieved.

In [ ]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages from the PDF.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)
print(f"Produced {len(chunks)} text chunks.")

# Peek at the first chunk
print("\n--- Sample chunk ---")
print(chunks[0].page_content[:400], "...")

## 6. Create vector embeddings and store them in Chroma

Each chunk is converted into a dense vector with OpenAI's `text-embedding-3-small` model and stored in an in-memory Chroma collection.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policy",
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Vector store built and retriever ready.")

## 7. Build the question-answering chain

We use **GPT-3.5 Turbo** with a custom prompt that:
* Keeps the assistant focused on Nestlé's HR policy.
* Forces it to answer only from the retrieved context.
* Falls back to a graceful "I don't know" instead of inventing facts.

The chain is built with **LCEL (LangChain Expression Language)** — the modern, recommended way to assemble RAG pipelines in LangChain 1.x.

In [ ]:
SYSTEM_PROMPT = """You are a helpful HR assistant for Nestlé employees.
Use the following pieces of context extracted from Nestlé's HR policy document
to answer the employee's question.

Guidelines:
- Answer only from the provided context. Do not use outside knowledge.
- If the answer is not contained in the context, say:
  "I'm sorry, I couldn't find that information in Nestlé's HR policy document."
- Be concise, professional, and structure longer answers as short bullet points.
- Quote the policy wording when helpful, but keep quotes brief.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)

def format_docs(docs):
    """Concatenate retrieved chunks into one context string."""
    return "\n\n".join(d.page_content for d in docs)

# LCEL RAG chain: retrieve -> format context -> prompt -> LLM -> parse text
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## 8. Quick sanity check

Try a few sample questions before launching the UI.

In [ ]:
sample_questions = [
    "What is Nestlé's policy on employee training and development?",
    "How does Nestlé approach recruitment and hiring?",
    "What does the policy say about work-life balance?",
]

for q in sample_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 80)

## 9. Build the Gradio chatbot UI

`gr.ChatInterface` gives us a polished chat window. Each user message is routed through the RAG chain, and retrieved source pages are appended at the bottom of every answer so users can verify the response.

Note for Gradio 6.x users: the `type` and `theme` keyword arguments were removed from `ChatInterface`. The messages API is the default; themes (if desired) are configured at the `Blocks` level.

In [ ]:
def answer_question(message, history):
    """Function called by Gradio for every user message."""
    if not message or not message.strip():
        return "Please type a question about Nestlé's HR policy."

    # Retrieve source documents separately so we can cite them.
    source_docs = retriever.invoke(message)
    answer = rag_chain.invoke(message)

    if source_docs:
        pages = sorted({doc.metadata.get("page", 0) + 1 for doc in source_docs})
        answer += f"\n\n_Sources: page(s) {', '.join(map(str, pages))} of the Nestlé HR policy._"

    return answer


demo = gr.ChatInterface(
    fn=answer_question,
    title="🍫 Nestlé HR Assistant",
    description=(
        "Ask any question about Nestlé's Human Resources policy. "
        "Powered by OpenAI GPT-3.5 Turbo, LangChain, and Chroma."
    ),
    examples=[
        "What is Nestlé's policy on diversity and inclusion?",
        "How does Nestlé handle employee performance management?",
        "What are the company's principles around employee relations?",
        "Tell me about Nestlé's approach to leadership development.",
        "What is the company's stance on work-life balance?",
    ],
)

# share=True gives you a public link that lasts 72 hours – useful for demo videos.
demo.launch(share=True, debug=False)

## 10. (Optional) Stop the server

Run the cell below if you want to free the port without restarting the runtime.

In [ ]:
# demo.close()

---
### Notes
* The Chroma collection lives in memory only; restart the runtime and re-run cells 5–6 to rebuild it.
* To swap models, change `gpt-3.5-turbo` to `gpt-4o-mini` (or another chat model) in the `ChatOpenAI` call.
* To use your own policy PDF, upload it to the Colab session and point `PDF_PATH` at it before running cell 5.

### Tested API surface (LangChain 1.x / Gradio 6.x)
Compared to older notebooks you may see online, the things that changed in the current stack:
* `langchain.text_splitter` ➜ `langchain_text_splitters`.
* `RetrievalQA` chain is deprecated; LCEL composition (`retriever | format_docs | prompt | llm | StrOutputParser`) is the supported pattern.
* `gr.ChatInterface` no longer accepts `type=...` or `theme=...` — the messages format is the default.

### About pip warnings
After cell 1 you may still see warnings like *"pip's dependency resolver does not currently take into account ..."* mentioning unrelated Colab-preinstalled packages (`dataproc-spark-connect`, `tobler`, `shap`, etc.). **Safe to ignore** — we never import those.